In [2]:
import polars as pl
from pathlib import Path

In [3]:
DATA_GENERAL = Path("../data_general")
DATA_PERSONAL = Path("../data_personal/Spotify Extended Streaming History")
OUT_DATA = Path('../output') 

In [4]:
lazy_general_df = pl.scan_parquet(DATA_GENERAL / "spotify_audio_features_*.parquet")

In [5]:
general_df = (
    lazy_general_df
    .filter(pl.col('null_response') == 0)   
    .drop('null_response')
    .collect()
)

In [6]:
general_df = general_df.rename({ 'id' : 'spotify_track_uri' })

In [7]:
display(general_df.head())

spotify_track_uri,name,popularity,duration_ms,time_signature,key,mode,tempo,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
str,str,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2Pe9cbhOTvOUTDE4bl7zzl""","""I dreamt you died""",0,630506,4,6,0,87.683,0.279,0.391,-12.054,0.32,0.816,0.737,0.177,0.0299
"""0wP732NKm8XgXu78XLRWoR""","""It's Death""",0,97216,4,5,1,105.298,0.429,0.318,-11.685,0.0566,0.587,0.782,0.202,0.36
"""22L6EJdnjx8oIo7GiF9hLe""","""Preliminary""",0,75180,4,0,1,117.657,0.283,0.581,-9.42,0.0555,0.923,0.939,0.106,0.0362
"""3a519lgQ13JXNi0G73mwMT""","""Disparage""",0,149447,4,5,0,100.685,0.244,0.995,-0.69,0.125,0.78,0.799,0.132,0.0634
"""27yP7p2lxWYTtnldRN8Kzx""","""Cut Down""",0,120816,4,7,1,123.499,0.313,0.618,0.411,0.073,0.843,0.109,0.126,0.187


In [8]:
personal_data_frames = {}
for num in range (2022,2027) :
    personal_data_frames[num] = pl.read_json(
        DATA_PERSONAL / f'Streaming_History_Audio_{num}.json',
        infer_schema_length=None  
        )

In [9]:
song_uri = '5v3Dv4UlDOGaXlvphFpkIX'

In [10]:
stats = general_df.filter(pl.col('spotify_track_uri') == song_uri)

In [11]:
display(stats)

spotify_track_uri,name,popularity,duration_ms,time_signature,key,mode,tempo,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
str,str,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""5v3Dv4UlDOGaXlvphFpkIX""","""Hello Juliet""",56,208000,4,3,1,138.991,0.334,0.992,-4.284,0.106,0.000171,0.111,0.153,0.227


In [12]:
# Subjectife paramaters (value given in percentage according to reverens value,
#  like 0.1 is mean : from 90% to 110% of reverense value)
cof_dance = 0.2
cof_energy = 0.2 
cof_speech = 0.2
cof_acoustic = 0.2
cof_instr = 0.2 
cof_live = 0.2
cof_valence = 0.2

# Technical paramaters (in absalute value)
cof_ts = 0
cof_key = 0
cof_mod = 0 
cof_tempo = 2 #bbm
cof_loud = 1 #Db

subj_features = [       
    ( cof_dance, 'danceability' ),     
    (cof_energy, 'energy'),    
    (cof_speech,  'speechiness' ),   
    (cof_acoustic,'acousticness') ,
    (cof_instr, 'instrumentalness'),    
    (cof_live, 'liveness'),    
    (cof_valence, 'valence'),     
]

tech_features = [ 
    (cof_tempo, 'tempo') ,         
    (cof_loud, 'loudness' ),    
    (cof_ts,  'time_signature' ),   
    (cof_key,'key') ,
    (cof_mod, 'mode'),     
]

In [13]:
conditions = []


In [14]:
# Accept subjective features

for cof, state_key in subj_features:
    if cof:
        state = stats[state_key]
        cond = pl.col(state_key).is_between(
            state * (1 - cof),
            state * (1 + cof)
        )
        conditions.append(cond)


In [15]:
# Accept techical features

for cof, state_key in tech_features:
    if cof:
        state = stats[state_key]
        cond = pl.col(state_key).is_between(
            state - cof,
            state + cof
        )
        conditions.append(cond)


In [16]:
if conditions:
    result = (
        general_df.lazy()
        .filter(conditions)
        .collect()
    )

In [17]:
print(len(result))
result = result.with_columns(
    pl.format("spotify:track:{}", pl.col("spotify_track_uri")).alias("spotify_track_uri")
)
display(result.head())


3


spotify_track_uri,name,popularity,duration_ms,time_signature,key,mode,tempo,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
str,str,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""spotify:track:6dJXKUEdEweErmQj…","""North of Nowhere""",0,310187,4,1,0,139.061,0.282,0.964,-5.1,0.0916,0.000188,0.091,0.147,0.222
"""spotify:track:6DBP4fIvKFRIshJr…","""Hello Juliet""",34,208000,4,3,1,139.14,0.332,0.992,-4.276,0.111,0.00017,0.114,0.146,0.212
"""spotify:track:5v3Dv4UlDOGaXlvp…","""Hello Juliet""",56,208000,4,3,1,138.991,0.334,0.992,-4.284,0.106,0.000171,0.111,0.153,0.227


In [18]:
file_path = OUT_DATA / f"Like '{stats['name'][0]}'.csv"
result.select('spotify_track_uri').write_csv(file_path , separator=',')